In [ ]:
"""
Interactive Behavior Feature Extraction Module
==============================================

This module provides lightweight utilities for detecting interactive
elements in LLM responses. These features help quantify whether a model
engages the user through: questions, conclusions, next-step guidance,
or invitations for interaction.

Extracted Features
------------------
- has_question
- has_conclusion
- has_next_steps
- has_interaction_prompt
- interaction_score  (0–4)

Usage
-----
from interaction_features import extract_interaction_features

df["interaction"] = df["response_text"].apply(extract_interaction_features)
"""

import re
from typing import Any, Dict


# ---------------------------------------------------------
# Helper: detect None / NaN / empty
# ---------------------------------------------------------
def _is_nan(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float) and x != x:
        return True
    return False


# ---------------------------------------------------------
# 1) Question Detection  
# ---------------------------------------------------------
def has_question(text: str, window: int = 3) -> bool:
    """
    Detect a real conversational question near the END of the text.
    Uses improved logic:
      - must contain '?'
      - must contain English letters (avoid non-English noise)
      - checks only last few lines
    """
    if _is_nan(text):
        return False

    lines = [l.strip() for l in str(text).split("\n") if l.strip()]
    if not lines:
        return False

    tail = "\n".join(lines[-window:]).lower()

    if "?" not in tail:
        return False

    if not re.search(r"[a-z]", tail):  
        return False

    question_patterns = [
        r"\?$",                  
        r"\bdo you\b",
        r"\bwould you\b",
        r"\bshould i\b",
        r"\bcan i\b",
        r"\bcan you\b",
        r"\bwhat else\b",
        r"\banything else\b",
        r"\bwant me to\b",
    ]

    return any(re.search(p, tail) for p in question_patterns)


# ---------------------------------------------------------
# 2) Conclusion Detection 
# ---------------------------------------------------------
def has_conclusion(text: str) -> bool:
    if _is_nan(text):
        return False

    t = str(text).lower().strip()

    paragraphs = [p.strip() for p in t.split("\n") if p.strip()]
    if not paragraphs:
        return False

    last_para = paragraphs[-1]

    patterns = [
        r"\bin conclusion\b",
        r"\bto conclude\b",
        r"\bto summarize\b",
        r"\bin summary\b",
        r"\bultimately\b",
        r"\bin the end\b",
        r"\bto wrap up\b",
        r"\btherefore\b",
        r"\boverall\b",
        r"^so[, ]",
    ]

    return any(re.search(p, last_para) for p in patterns)


# ---------------------------------------------------------
# 3) Next-step Guidance Detection
# ---------------------------------------------------------
def has_next_steps(text: str, para_window: int = 3) -> bool:
    """
    Detect actionable next-step guidance with:
      - second-person pronouns ('you', 'your')
      - directive patterns (you can / should / consider / try...)
      - restricted to last N paragraphs
    """
    if _is_nan(text):
        return False

    t = str(text).strip().lower()
    paragraphs = [p.strip() for p in t.split("\n") if p.strip()]
    if not paragraphs:
        return False

    tail = "\n".join(paragraphs[-para_window:])

   
    if not re.search(r"\b(you|your)\b", tail):
        return False

    patterns = [
        r"\byou can\b",
        r"\byou should\b",
        r"\bwhat you can do\b",
        r"\bhere(?:'s| is) what you\b",
        r"\bnext[, ]?\b",
        r"\brecommend( you)?\b",
        r"\bconsider\b",
        r"\btry\b",
        r"\bto do this, you\b",
        r"\bthe next step\b",
    ]

    return any(re.search(p, tail) for p in patterns)


# ---------------------------------------------------------
# 4) Interaction Invitation Detection 
# ---------------------------------------------------------
def has_interaction_prompt(text: str) -> bool:
    if _is_nan(text):
        return False

    t = str(text).lower()

    patterns = [
        r"\blet me know\b",
        r"\bfeel free to ask\b",
        r"\bif you'd like\b",
        r"\bif you want\b",
        r"\bwould you like\b",
        r"\bany other questions\b",
        r"\banything else\b",
    ]

    return any(re.search(p, t) for p in patterns)


# ---------------------------------------------------------
# Master Extractor — same API as original version
# ---------------------------------------------------------
def extract_interaction_features(text: Any) -> Dict[str, Any]:
    """
    Return:
      - has_question
      - has_conclusion
      - has_next_steps
      - has_interaction_prompt
      - interaction_score
    """
    t = "" if _is_nan(text) else str(text)

    q = has_question(t)
    c = has_conclusion(t)
    n = has_next_steps(t)
    p = has_interaction_prompt(t)

    score = int(q) + int(c) + int(n) + int(p)

    return {
        "has_question": q,
        "has_conclusion": c,
        "has_next_steps": n,
        "has_interaction_prompt": p,
        "interaction_score": score,
    }


__all__ = [
    "extract_interaction_features",
    "has_question",
    "has_conclusion",
    "has_next_steps",
    "has_interaction_prompt",
]